# Notebook 03 - SFT Without Adaption

This is the raw-data SFT condition. It trains the base model on `data/processed/kakugo_raw_sft.jsonl`.

Notebook 04 mirrors this notebook. The only intended difference is the training data path.

**Files read:**
- [`../configs/tinker_sft_raw.yaml`](../configs/tinker_sft_raw.yaml) - base model, renderer, training hyperparameters, input path, and output path for the raw-data SFT run.
- [`../data/processed/kakugo_raw_sft.jsonl`](../data/processed/kakugo_raw_sft.jsonl) - chat-format raw SFT examples prepared in Notebook 01.

**Files written:**
- [`../results/models/sft_raw/`](../results/models/sft_raw/) - Tinker logs, metrics, checkpoints, and final model/sampler references for the raw-data SFT run.

In [ ]:
import asyncio
import json
import logging
import os
import sys
import warnings
from pathlib import Path

import nest_asyncio
from huggingface_hub import login
from tinker_cookbook.supervised.data import FromConversationFileBuilder
from tinker_cookbook.supervised.train import Config, main
from tinker_cookbook.supervised.types import ChatDatasetBuilderCommonConfig

nest_asyncio.apply()
warnings.filterwarnings('ignore', module='tinker_cookbook')

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )


ROOT = find_repo_root(Path.cwd())
SRC = str(ROOT / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

In [ ]:
from kirundi_sft_starter.utils import ensure_dir, load_env, load_yaml, require_file

load_env()
raw_sft_config = load_yaml(ROOT / 'configs/tinker_sft_raw.yaml')

require_file(ROOT / raw_sft_config['data_path'], 'Run Notebook 01 before launching raw-data SFT.')
ensure_dir(ROOT / raw_sft_config['output_dir'])

if not os.environ.get('TINKER_API_KEY'):
    raise RuntimeError('Missing TINKER_TOKEN or TINKER_API_KEY. Add it to .env before launching training.')

if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'])

## Review the training config

This is the exact raw-data SFT configuration that will be sent to Tinker in the training cell below.

In [ ]:
print(json.dumps(raw_sft_config, indent=2))

## Build the Tinker SFT config

The dataset builder reads the chat-format JSONL created in Notebook 01. `train_on_what='last_assistant_message'` means the supervised loss is applied to the assistant response, not the user prompt.

In [ ]:
renderer_name = raw_sft_config['renderer_name']

common_config = ChatDatasetBuilderCommonConfig(
    model_name_for_tokenizer=raw_sft_config['base_model'],
    renderer_name=renderer_name,
    max_length=int(raw_sft_config['max_length']),
    batch_size=int(raw_sft_config['batch_size']),
    train_on_what=raw_sft_config['train_on_what'],
)

dataset_builder = FromConversationFileBuilder(
    file_path=str(ROOT / raw_sft_config['data_path']),
    common_config=common_config,
)

sft_config = Config(
    log_path=str(ROOT / raw_sft_config['output_dir']),
    model_name=raw_sft_config['base_model'],
    dataset_builder=dataset_builder,
    learning_rate=float(raw_sft_config['learning_rate']),
    lora_rank=int(raw_sft_config['lora_rank']),
    num_epochs=int(raw_sft_config['num_epochs']),
)

print('\nSFT Config:')
print(f"  Run:           {raw_sft_config['run_name']}")
print(f"  Model:         {sft_config.model_name}")
print(f"  Renderer:      {renderer_name}")
print(f"  Data:          {ROOT / raw_sft_config['data_path']}")
print(f"  Output:        {sft_config.log_path}")
print(f"  Learning rate: {sft_config.learning_rate}")
print(f"  LoRA rank:     {sft_config.lora_rank}")
print(f"  Epochs:        {sft_config.num_epochs}")

## Launch training

This cell starts the real Tinker SFT job. If credentials, package imports, data paths, or Tinker access are wrong, the cell should fail loudly so the issue is visible.

In [ ]:
print('=' * 50)
print('Starting raw-data SFT training...')
print('Watch train_mean_nll; it should generally decrease across training.\n')

logging.getLogger('asyncio').setLevel(logging.CRITICAL)
result = asyncio.get_event_loop().run_until_complete(main(sft_config))

print('\nTraining complete!')
print(f'Result: {result}')